# 📚 WikiText-103 Preprocessing Pipeline

This code cleans and prepares the **WikiText-103 dataset** for NLP model training.

---

## 🎯 Goal
Convert raw text into **clean, meaningful sentences** by removing noise and fixing formatting.

---

## ⚙️ Steps
- Load dataset  
- Fix special tokens (`@-@`, `<unk>`)  
- Remove section titles and short lines  
- Remove non-English characters  
- Clean spacing and punctuation  
- Split into sentences  
- Filter noisy sentences  
- Shuffle and save output  

---

## 📊 Output
- File: `clean_wikitext.txt`  
- Format: One sentence per line  
- Ready for training ✅

In [2]:
import re
import nltk
import random
nltk.download("punkt")
from nltk.tokenize import sent_tokenize

# =========================
# STEP 1: LOAD DATA
# =========================
with open("/kaggle/input/datasets/vadimkurochkin/wikitext-103/wikitext-103/wiki.train.tokens", "r", encoding="utf-8") as f:
    text = f.read()

print("\n[INFO] Loaded dataset")
print("Total characters:", len(text))
print("Sample raw text:\n", text[:300])


# =========================
# STEP 2: BASIC CLEANING
# =========================

# Fix special tokens
text = text.replace("@-@", "-")
text = text.replace("@,@", ",")
text = text.replace("<unk>", "UNK")

print("\n[INFO] After fixing special tokens")
print(text[:300])


# =========================
# STEP 3: REMOVE SECTION TITLES
# =========================
text = re.sub(r"=+.*?=+", " ", text)

print("\n[INFO] After removing section titles")
print(text[:300])


# =========================
# STEP 4: REMOVE HEADINGS / SHORT LINES
# =========================
lines = text.split("\n")
print("\n[INFO] Total lines before cleaning:", len(lines))

clean_lines = []
for line in lines:
    line = line.strip()
    
    # remove very short / title-like lines
    if len(line.split()) < 5:
        continue
    
    clean_lines.append(line)

print("[INFO] Lines after cleaning:", len(clean_lines))

# join lines to avoid broken sentences
text = " ".join(clean_lines)


# =========================
# STEP 5: REMOVE NON-ENGLISH / SYMBOLS
# =========================
text = re.sub(r"[^a-zA-Z0-9.,!?;:'\"()\-\s]", " UNK ", text)

print("\n[INFO] After removing non-English characters")
print(text[:300])


# =========================
# STEP 6: IMPORTANT FIXES
# =========================

# collapse multiple UNK → single UNK
text = re.sub(r"(UNK\s+)+UNK", "UNK", text)

# fix UNKword issue
text = re.sub(r"UNK([a-zA-Z])", r"UNK \1", text)

# fix hyphen spacing
text = re.sub(r"\s*-\s*", "-", text)

# fix possessive
text = re.sub(r"\s+'s", "'s", text)

# remove repeated punctuation
text = re.sub(r"([.!?])\1+", r"\1", text)

# remove leftover newlines
text = text.replace("\n", " ")

print("\n[INFO] After advanced fixes")
print(text[:300])


# =========================
# STEP 7: CLEAN SPACING
# =========================

text = re.sub(r"\s+", " ", text)

text = re.sub(r"\s+([.,!?;:])", r"\1", text)
text = re.sub(r"([.,!?;:])([a-zA-Z])", r"\1 \2", text)

print("\n[INFO] After spacing cleanup")
print(text[:300])


# =========================
# STEP 8: SENTENCE SPLITTING
# =========================

sentences = sent_tokenize(text)

print("\n[INFO] Total sentences:", len(sentences))


# =========================
# STEP 9: FILTER SENTENCES
# =========================

clean_sentences = []

for s in sentences:
    s = s.strip()
    
    if 5 < len(s) < 300:
        # remove sentences with too much UNK noise
        if s.count("UNK") / len(s.split()) < 0.3:
            clean_sentences.append(s)

# shuffle (VERY IMPORTANT for training)
random.shuffle(clean_sentences)

print("[INFO] Sentences after filtering:", len(clean_sentences))


# =========================
# STEP 10: SAVE CLEAN DATA
# =========================

with open("clean_wikitext.txt", "w", encoding="utf-8") as f:
    for s in clean_sentences:
        f.write(s + "\n")

print("\n[INFO] Saved cleaned dataset to clean_wikitext.txt")


# =========================
# STEP 11: PRINT SAMPLE OUTPUT
# =========================

print("\n===== SAMPLE SENTENCES =====\n")
for i in range(10):
    print(clean_sentences[i])

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!



[INFO] Loaded dataset
Total characters: 538360726
Sample raw text:
  
 = Valkyria Chronicles III = 
 
 Senjō no Valkyria 3 : <unk> Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStatio

[INFO] After fixing special tokens
 
 = Valkyria Chronicles III = 
 
 Senjō no Valkyria 3 : UNK Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role - playing video game developed by Sega and Media.Vision for the PlayStation Po

[INFO] After removing section titles
 
   
 
 Senjō no Valkyria 3 : UNK Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role - playing video game developed by Sega and Media.Vision for

# 🤖 Mini GPT (Character-Level) Training

This code trains a simple **character-level GPT model** on cleaned WikiText data.

---

## 🎯 Goal
Learn patterns in text and generate new text using a Transformer model.

---

## ⚙️ Steps
- Load cleaned dataset  
- Convert characters to tokens  
- Create train/validation split  
- Build Transformer (self-attention + feedforward)  
- Train with early stopping  
- Generate new text  

---

## 📊 Output
- Trained model: `best_model.pt`  
- Generated text samples  

In [ ]:
# ================================
# MINI GPT (CHAR-LEVEL) + EARLY STOPPING
# ================================

import torch
import torch.nn as nn
from torch.nn import functional as F

# ================================
# 1. LOAD CLEAN DATA (FIXED ✅)
# ================================

with open("clean_wikitext.txt", "r", encoding="utf-8") as f:
    clean_sentences = f.readlines()

# join into single text (VERY IMPORTANT)
text = " ".join([s.strip() for s in clean_sentences])

print("Dataset size (chars):", len(text))
print("Sample cleaned text:\n", text[:300])


# ================================
# 2. CHAR TOKENIZATION (UNCHANGED)
# ================================

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return "".join([itos[i] for i in l])


# ================================
# 3. DATA TO TENSOR (UNCHANGED)
# ================================

data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


# ================================
# 4. HYPERPARAMETERS (UNCHANGED)
# ================================

batch_size = 32
block_size = 256
max_iters = 70000
eval_interval = 500
learning_rate = 3e-4

n_embd = 256
n_head = 4
n_layer = 4
dropout = 0.2

device = "cuda" if torch.cuda.is_available() else "cpu"


# ================================
# 5. BATCH LOADER (UNCHANGED)
# ================================

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)


# ================================
# 6. TRANSFORMER COMPONENTS (UNCHANGED)
# ================================

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        return wei @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


# ================================
# 7. GPT MODEL (UNCHANGED)
# ================================

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))

        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            logits = logits.view(B*T, -1)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx


# ================================
# 8. LOSS ESTIMATION (UNCHANGED)
# ================================

@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}

    for split in ["train", "val"]:
        loss_list = []
        for _ in range(20):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            loss_list.append(loss.item())

        losses[split] = sum(loss_list) / len(loss_list)

    model.train()
    return losses


# ================================
# 9. TRAINING + EARLY STOPPING (UNCHANGED)
# ================================

model = GPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

best_val_loss = float("inf")
patience = 5
counter = 0

for step in range(max_iters):

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0:

        losses = estimate_loss()
        print(f"step {step}, train {losses['train']:.4f}, val {losses['val']:.4f}")

        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            counter = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            counter += 1

        if counter >= patience:
            print("🚀 Early stopping triggered!")
            break


# ================================
# 10. GENERATE TEXT
# ================================

context = torch.zeros((1, 1), dtype=torch.long).to(device)
generated = model.generate(context, max_new_tokens=500)

print("\n=== GENERATED TEXT ===\n")
print(decode(generated[0].tolist()))

Dataset size (chars): 478328251
Sample cleaned text:
 Despite this, Sorkin sent Fey flowers after NBC announced it would pick up both programs, and wished her luck with 30 Rock. Digital Spy ranked the song No. The earliest predecessor of the Cabinet was arguably the Executive Council of the Straits Settlements that was introduced in 1877 by letters pat
step 0, train 3.9837, val 3.9851
step 500, train 2.4587, val 2.4631
step 1000, train 2.1937, val 2.1936
step 1500, train 1.9895, val 1.9775
step 2000, train 1.8409, val 1.8452
step 2500, train 1.7411, val 1.7500
step 3000, train 1.6762, val 1.6703
step 3500, train 1.6272, val 1.6282
step 4000, train 1.5787, val 1.5839
step 4500, train 1.5543, val 1.5575
step 5000, train 1.5268, val 1.5274
step 5500, train 1.5038, val 1.5121
step 6000, train 1.4874, val 1.4866
step 6500, train 1.4800, val 1.4691
step 7000, train 1.4683, val 1.4638
step 7500, train 1.4486, val 1.4523
step 8000, train 1.4336, val 1.4434
step 8500, train 1.4306, val 1.4442
s

# 🔄 Improvements Over Previous Version

This version introduces several upgrades compared to the earlier **character-level GPT model**.

---

## ✨ Key Changes

- **🔤 Tokenization**
  - Before: Character-level (each character = token)
  - Now: BPE tokenization using GPT-2 tokenizer  
  👉 Learns subword units → better language understanding

- **⚡ Chunked Encoding**
  - Encodes large text in chunks  
  👉 Prevents memory issues during tokenization

- **📉 Data Limiting**
  - Uses first 5M characters  
  👉 Faster training and experimentation

- **📦 Smaller Input Configuration**
  - Reduced `block_size` and `batch_size`  
  👉 Improves training efficiency and memory usage

- **🎯 Improved Generation Setup**
  - Uses BOS/EOS token instead of zero initialization  
  👉 More meaningful starting context

---

## 🚀 Overall Benefit

- More meaningful token representation (subword-level)  
- Faster and more efficient training  
- Better quality text generation compared to char-level  

In [4]:
# ================================
# MINI GPT (BPE + CHUNKED + EARLY STOPPING)
# ================================

import torch
import torch.nn as nn
from torch.nn import functional as F
from transformers import GPT2Tokenizer

# ================================
# 1. LOAD CLEAN DATA (FIXED ✅)
# ================================

with open("clean_wikitext.txt", "r", encoding="utf-8") as f:
    clean_sentences = f.readlines()

# join into continuous text
text = " ".join([s.strip() for s in clean_sentences])[:5_000_000]

print("Dataset size (chars used):", len(text))
print("Sample cleaned text:\n", text[:300])


# ================================
# 2. BPE TOKENIZER (UNCHANGED)
# ================================

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

vocab_size = tokenizer.vocab_size

def encode(s):
    return tokenizer.encode(s)

def decode(l):
    return tokenizer.decode(l)


# ================================
# 3. CHUNKED ENCODING (UNCHANGED)
# ================================

def encode_chunked(text, chunk_size=100000):
    tokens = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i+chunk_size]
        tokens.extend(tokenizer.encode(chunk))
    return tokens

print("Encoding text in chunks...")
encoded = encode_chunked(text)

# 🔥 FIX: use long (VERY IMPORTANT)
data = torch.tensor(encoded, dtype=torch.long)


# ================================
# 4. TRAIN / VAL SPLIT (UNCHANGED)
# ================================

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


# ================================
# 5. HYPERPARAMETERS (UNCHANGED)
# ================================

batch_size = 16
block_size = 128
max_iters = 70000
eval_interval = 500
learning_rate = 3e-4

n_embd = 256
n_head = 4
n_layer = 4
dropout = 0.2

device = "cuda" if torch.cuda.is_available() else "cpu"


# ================================
# 6. BATCH LOADER (UNCHANGED)
# ================================

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)


# ================================
# 7. TRANSFORMER (UNCHANGED)
# ================================

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        return wei @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))


class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))

        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            logits = logits.view(B*T, -1)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx


# ================================
# TRAINING LOOP (UNCHANGED)
# ================================

@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}
    for split in ["train", "val"]:
        loss_list = []
        for _ in range(20):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            loss_list.append(loss.item())
        losses[split] = sum(loss_list) / len(loss_list)
    model.train()
    return losses


model = GPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

best_val_loss = float("inf")
patience = 5
counter = 0

for step in range(max_iters):

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0:

        losses = estimate_loss()
        print(f"step {step}, train {losses['train']:.4f}, val {losses['val']:.4f}")

        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            counter = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            counter += 1

        if counter >= patience:
            print("🚀 Early stopping triggered!")
            break


# ================================
# GENERATION
# ================================

start_token = tokenizer.bos_token_id or tokenizer.eos_token_id
context = torch.tensor([[start_token]], device=device)

generated = model.generate(context, max_new_tokens=300)

print("\n=== GENERATED TEXT ===\n")
print(decode(generated[0].tolist()))

Dataset size (chars used): 5000000
Sample cleaned text:
 Despite this, Sorkin sent Fey flowers after NBC announced it would pick up both programs, and wished her luck with 30 Rock. Digital Spy ranked the song No. The earliest predecessor of the Cabinet was arguably the Executive Council of the Straits Settlements that was introduced in 1877 by letters pat


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (21431 > 1024). Running this sequence through the model will result in indexing errors


Encoding text in chunks...
step 0, train 10.8001, val 10.7991
step 500, train 6.8125, val 6.9299
step 1000, train 6.4064, val 6.6295
step 1500, train 6.1002, val 6.4594
step 2000, train 5.8832, val 6.3091
step 2500, train 5.6774, val 6.2620
step 3000, train 5.4885, val 6.1968
step 3500, train 5.3780, val 6.1461
step 4000, train 5.1748, val 6.1495
step 4500, train 5.0365, val 6.1291
step 5000, train 4.9103, val 6.1329
step 5500, train 4.7966, val 6.1627
step 6000, train 4.7025, val 6.1699
step 6500, train 4.6120, val 6.1600
step 7000, train 4.4888, val 6.1979
🚀 Early stopping triggered!

=== GENERATED TEXT ===

<|endoftext|>ere and Dr. Melinault UNK 1 percentage, 900 million outscored innocent on the Top Army and the Robertson West Coast roofed, VorM-VS Poles title site. The shots of Bisexual the Germans as a maximum of 25 the promotional meeting, 676, 000 copies. On the court injunction 3, 1805ers repeating, Carlt's Baby. and Sok Nath made Roman Catholic Church of the fourth consecutiv

#  Pipeline

Below version combines **data cleaning, sentence preprocessing, BPE tokenization, and dataset analysis** in one pipeline.

---

## ✨ What’s New

- End-to-end pipeline (raw → clean → sentence-level data → tokenize → analyze)
- Advanced preprocessing:
  - remove section titles and noisy lines  
  - filter low-quality sentences (UNK-heavy)  
- Added dataset statistics:
  - total words and BPE tokens  
  - unique words and unique tokens  
  - avg tokens per word, avg word length  
- Uses GPT-2 BPE tokenizer (better than char-level)
- Chunked encoding for large data handling
- Limited dataset size for faster processing

---

## 🚀 Benefit

- Cleaner and higher-quality training data  
- Better token representation using subwords  
- Useful insights into dataset before training  
- More efficient and scalable pipeline  

In [1]:

import re
import nltk
import random
nltk.download("punkt")
from nltk.tokenize import sent_tokenize

# =========================
# STEP 1: LOAD DATA
# =========================
with open("/kaggle/input/datasets/vadimkurochkin/wikitext-103/wikitext-103/wiki.train.tokens", "r", encoding="utf-8") as f:
    text = f.read()

print("\n[INFO] Loaded dataset")
print("Total characters:", len(text))
print("Sample raw text:\n", text[:300])


# =========================
# STEP 2: BASIC CLEANING
# =========================

# Fix special tokens
text = text.replace("@-@", "-")
text = text.replace("@,@", ",")
text = text.replace("<unk>", "UNK")

print("\n[INFO] After fixing special tokens")
print(text[:300])


# =========================
# STEP 3: REMOVE SECTION TITLES
# =========================
text = re.sub(r"=+.*?=+", " ", text)

print("\n[INFO] After removing section titles")
print(text[:300])


# =========================
# STEP 4: REMOVE HEADINGS / SHORT LINES
# =========================
lines = text.split("\n")
print("\n[INFO] Total lines before cleaning:", len(lines))

clean_lines = []
for line in lines:
    line = line.strip()
    
    # remove very short / title-like lines
    if len(line.split()) < 5:
        continue
    
    clean_lines.append(line)

print("[INFO] Lines after cleaning:", len(clean_lines))

# join lines to avoid broken sentences
text = " ".join(clean_lines)


# =========================
# STEP 5: REMOVE NON-ENGLISH / SYMBOLS
# =========================
text = re.sub(r"[^a-zA-Z0-9.,!?;:'\"()\-\s]", " UNK ", text)

print("\n[INFO] After removing non-English characters")
print(text[:300])


# =========================
# STEP 6: IMPORTANT FIXES
# =========================

# collapse multiple UNK → single UNK
text = re.sub(r"(UNK\s+)+UNK", "UNK", text)

# fix UNKword issue
text = re.sub(r"UNK([a-zA-Z])", r"UNK \1", text)

# fix hyphen spacing
text = re.sub(r"\s*-\s*", "-", text)

# fix possessive
text = re.sub(r"\s+'s", "'s", text)

# remove repeated punctuation
text = re.sub(r"([.!?])\1+", r"\1", text)

# remove leftover newlines
text = text.replace("\n", " ")

print("\n[INFO] After advanced fixes")
print(text[:300])


# =========================
# STEP 7: CLEAN SPACING
# =========================

text = re.sub(r"\s+", " ", text)

text = re.sub(r"\s+([.,!?;:])", r"\1", text)
text = re.sub(r"([.,!?;:])([a-zA-Z])", r"\1 \2", text)

print("\n[INFO] After spacing cleanup")
print(text[:300])


# =========================
# STEP 8: SENTENCE SPLITTING
# =========================

sentences = sent_tokenize(text)

print("\n[INFO] Total sentences:", len(sentences))


# =========================
# STEP 9: FILTER SENTENCES
# =========================

clean_sentences = []

for s in sentences:
    s = s.strip()
    
    if 5 < len(s) < 300:
        # remove sentences with too much UNK noise
        if s.count("UNK") / len(s.split()) < 0.3:
            clean_sentences.append(s)

# shuffle (VERY IMPORTANT for training)
random.shuffle(clean_sentences)

print("[INFO] Sentences after filtering:", len(clean_sentences))


# =========================
# STEP 10: SAVE CLEAN DATA
# =========================

with open("clean_wikitext.txt", "w", encoding="utf-8") as f:
    for s in clean_sentences:
        f.write(s + "\n")

print("\n[INFO] Saved cleaned dataset to clean_wikitext.txt")


# =========================
# STEP 11: PRINT SAMPLE OUTPUT
# =========================

print("\n===== SAMPLE SENTENCES =====\n")
for i in range(10):
    print(clean_sentences[i])


import torch
import torch.nn as nn
from torch.nn import functional as F
from transformers import GPT2Tokenizer

# ================================
# 1. LOAD CLEAN DATA (FIXED ✅)
# ================================

with open("clean_wikitext.txt", "r", encoding="utf-8") as f:
    clean_sentences = f.readlines()

# join into continuous text
text = " ".join([s.strip() for s in clean_sentences])[:5_000_000]

print("Dataset size (chars used):", len(text))
print("Sample cleaned text:\n", text[:300])


# ================================
# 2. BPE TOKENIZER (UNCHANGED)
# ================================

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

vocab_size = tokenizer.vocab_size

def encode(s):
    return tokenizer.encode(s)

def decode(l):
    return tokenizer.decode(l)


# ================================
# 3. CHUNKED ENCODING (UNCHANGED)
# ================================

def encode_chunked(text, chunk_size=100000):
    tokens = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i+chunk_size]
        tokens.extend(tokenizer.encode(chunk))
    return tokens

print("Encoding text in chunks...")
encoded = encode_chunked(text)

# 🔥 FIX: use long (VERY IMPORTANT)
data = torch.tensor(encoded, dtype=torch.long)

# ================================
# 📊 DATA ANALYSIS: WORDS & TOKENS
# ================================

print("\n===== DATASET ANALYSIS =====")

# 🔹 1. WORD COUNT
# simple whitespace split
words = text.split()
num_words = len(words)

# 🔹 2. TOKEN COUNT (BPE)
num_tokens = len(encoded)

# 🔹 3. UNIQUE WORDS
unique_words = len(set(words))

# 🔹 4. UNIQUE TOKENS
unique_tokens = len(set(encoded))

# 🔹 5. AVG TOKENS PER WORD
avg_tokens_per_word = num_tokens / num_words

# 🔹 6. AVG WORD LENGTH
avg_word_len = sum(len(w) for w in words) / num_words

# 🔹 7. PRINT RESULTS
print(f"Total Words           : {num_words:,}")
print(f"Total Tokens (BPE)    : {num_tokens:,}")
print(f"Unique Words          : {unique_words:,}")
print(f"Unique Tokens         : {unique_tokens:,}")
print(f"Avg Tokens per Word   : {avg_tokens_per_word:.2f}")
print(f"Avg Word Length       : {avg_word_len:.2f}")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!



[INFO] Loaded dataset
Total characters: 538360726
Sample raw text:
  
 = Valkyria Chronicles III = 
 
 Senjō no Valkyria 3 : <unk> Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStatio

[INFO] After fixing special tokens
 
 = Valkyria Chronicles III = 
 
 Senjō no Valkyria 3 : UNK Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role - playing video game developed by Sega and Media.Vision for the PlayStation Po

[INFO] After removing section titles
 
   
 
 Senjō no Valkyria 3 : UNK Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role - playing video game developed by Sega and Media.Vision for

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (21406 > 1024). Running this sequence through the model will result in indexing errors


Encoding text in chunks...

===== DATASET ANALYSIS =====
Total Words           : 854,588
Total Tokens (BPE)    : 1,070,590
Unique Words          : 92,645
Unique Tokens         : 35,379
Avg Tokens per Word   : 1.25
Avg Word Length       : 4.85


# 🔄 Improved GPT Pipeline

This version builds on the previous pipeline by introducing a **larger Transformer model and improved text generation strategies**.

---

## ✨ Key Improvements

- **🧠 Larger Model Capacity**
  - Increased layers, attention heads, and embedding size  
  👉 Learns more complex language patterns

- **📏 Longer Context Window**
  - Increased `block_size` (256)  
  👉 Captures longer dependencies in text

- **⚡ Improved Training Setup**
  - Reduced iterations (20k) with stronger model  
  - Lower dropout for better learning  
  👉 Faster convergence with improved performance

- **🎯 Advanced Text Generation**
  - Added:
    - Temperature scaling  
    - Top-k sampling  
  👉 Better control over randomness and diversity

- **📊 Full End-to-End Pipeline**
  - Includes preprocessing + BPE tokenization + dataset analysis + training  
  👉 Complete language modeling workflow

---

## 🚀 Overall Benefit

- Higher quality and more coherent text generation  
- Stronger model representation  
- Better control over generated outputs  
- Scalable end-to-end NLP pipeline  

In [2]:

import re
import nltk
import random
nltk.download("punkt")
from nltk.tokenize import sent_tokenize

# =========================
# STEP 1: LOAD DATA
# =========================
with open("/kaggle/input/datasets/vadimkurochkin/wikitext-103/wikitext-103/wiki.train.tokens", "r", encoding="utf-8") as f:
    text = f.read()

print("\n[INFO] Loaded dataset")
print("Total characters:", len(text))
print("Sample raw text:\n", text[:300])


# =========================
# STEP 2: BASIC CLEANING
# =========================

# Fix special tokens
text = text.replace("@-@", "-")
text = text.replace("@,@", ",")
text = text.replace("<unk>", "UNK")

print("\n[INFO] After fixing special tokens")
print(text[:300])


# =========================
# STEP 3: REMOVE SECTION TITLES
# =========================
text = re.sub(r"=+.*?=+", " ", text)

print("\n[INFO] After removing section titles")
print(text[:300])


# =========================
# STEP 4: REMOVE HEADINGS / SHORT LINES
# =========================
lines = text.split("\n")
print("\n[INFO] Total lines before cleaning:", len(lines))

clean_lines = []
for line in lines:
    line = line.strip()
    
    # remove very short / title-like lines
    if len(line.split()) < 5:
        continue
    
    clean_lines.append(line)

print("[INFO] Lines after cleaning:", len(clean_lines))

# join lines to avoid broken sentences
text = " ".join(clean_lines)


# =========================
# STEP 5: REMOVE NON-ENGLISH / SYMBOLS
# =========================
text = re.sub(r"[^a-zA-Z0-9.,!?;:'\"()\-\s]", " UNK ", text)

print("\n[INFO] After removing non-English characters")
print(text[:300])


# =========================
# STEP 6: IMPORTANT FIXES
# =========================

# collapse multiple UNK → single UNK
text = re.sub(r"(UNK\s+)+UNK", "UNK", text)

# fix UNKword issue
text = re.sub(r"UNK([a-zA-Z])", r"UNK \1", text)

# fix hyphen spacing
text = re.sub(r"\s*-\s*", "-", text)

# fix possessive
text = re.sub(r"\s+'s", "'s", text)

# remove repeated punctuation
text = re.sub(r"([.!?])\1+", r"\1", text)

# remove leftover newlines
text = text.replace("\n", " ")

print("\n[INFO] After advanced fixes")
print(text[:300])


# =========================
# STEP 7: CLEAN SPACING
# =========================

text = re.sub(r"\s+", " ", text)

text = re.sub(r"\s+([.,!?;:])", r"\1", text)
text = re.sub(r"([.,!?;:])([a-zA-Z])", r"\1 \2", text)

print("\n[INFO] After spacing cleanup")
print(text[:300])


# =========================
# STEP 8: SENTENCE SPLITTING
# =========================

sentences = sent_tokenize(text)

print("\n[INFO] Total sentences:", len(sentences))


# =========================
# STEP 9: FILTER SENTENCES
# =========================

clean_sentences = []

for s in sentences:
    s = s.strip()
    
    if 5 < len(s) < 300:
        # remove sentences with too much UNK noise
        if s.count("UNK") / len(s.split()) < 0.3:
            clean_sentences.append(s)

# shuffle (VERY IMPORTANT for training)
random.shuffle(clean_sentences)

print("[INFO] Sentences after filtering:", len(clean_sentences))


# =========================
# STEP 10: SAVE CLEAN DATA
# =========================

with open("clean_wikitext.txt", "w", encoding="utf-8") as f:
    for s in clean_sentences:
        f.write(s + "\n")

print("\n[INFO] Saved cleaned dataset to clean_wikitext.txt")


# =========================
# STEP 11: PRINT SAMPLE OUTPUT
# =========================

print("\n===== SAMPLE SENTENCES =====\n")
for i in range(10):
    print(clean_sentences[i])


import torch
import torch.nn as nn
from torch.nn import functional as F
from transformers import GPT2Tokenizer

# ================================
# 1. LOAD CLEAN DATA (FIXED ✅)
# ================================

with open("clean_wikitext.txt", "r", encoding="utf-8") as f:
    clean_sentences = f.readlines()

# join into continuous text
text = " ".join([s.strip() for s in clean_sentences])[:5_000_000]

print("Dataset size (chars used):", len(text))
print("Sample cleaned text:\n", text[:300])


# ================================
# 2. BPE TOKENIZER (UNCHANGED)
# ================================

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

vocab_size = tokenizer.vocab_size

def encode(s):
    return tokenizer.encode(s)

def decode(l):
    return tokenizer.decode(l)


# ================================
# 3. CHUNKED ENCODING (UNCHANGED)
# ================================

def encode_chunked(text, chunk_size=100000):
    tokens = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i+chunk_size]
        tokens.extend(tokenizer.encode(chunk))
    return tokens

print("Encoding text in chunks...")
encoded = encode_chunked(text)

# 🔥 FIX: use long (VERY IMPORTANT)
data = torch.tensor(encoded, dtype=torch.long)

# ================================
# 📊 DATA ANALYSIS: WORDS & TOKENS
# ================================

print("\n===== DATASET ANALYSIS =====")

# 🔹 1. WORD COUNT
# simple whitespace split
words = text.split()
num_words = len(words)

# 🔹 2. TOKEN COUNT (BPE)
num_tokens = len(encoded)

# 🔹 3. UNIQUE WORDS
unique_words = len(set(words))

# 🔹 4. UNIQUE TOKENS
unique_tokens = len(set(encoded))

# 🔹 5. AVG TOKENS PER WORD
avg_tokens_per_word = num_tokens / num_words

# 🔹 6. AVG WORD LENGTH
avg_word_len = sum(len(w) for w in words) / num_words

# 🔹 7. PRINT RESULTS
print(f"Total Words           : {num_words:,}")
print(f"Total Tokens (BPE)    : {num_tokens:,}")
print(f"Unique Words          : {unique_words:,}")
print(f"Unique Tokens         : {unique_tokens:,}")
print(f"Avg Tokens per Word   : {avg_tokens_per_word:.2f}")
print(f"Avg Word Length       : {avg_word_len:.2f}")


# ================================
# 4. TRAIN / VAL SPLIT (UNCHANGED)
# ================================

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


batch_size = 32          # bigger batch
block_size = 256         # longer context
max_iters = 20000        # you don't need 70k
learning_rate = 3e-4

n_embd = 512             # ⬆️ bigger
n_head = 8               # ⬆️ more heads
n_layer = 8              # ⬆️ deeper
dropout = 0.1            # slightly lower
eval_interval = 500

device = "cuda" if torch.cuda.is_available() else "cpu"


# ================================
# 6. BATCH LOADER (UNCHANGED)
# ================================

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)


# ================================
# 7. TRANSFORMER (UNCHANGED)
# ================================

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        return wei @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))


class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))

        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            logits = logits.view(B*T, -1)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
    
            logits = logits[:, -1, :] / temperature
    
            # top-k filtering
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("Inf")
    
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
    
            idx = torch.cat((idx, idx_next), dim=1)
    
        return idx


# ================================
# TRAINING LOOP (UNCHANGED)
# ================================

@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}
    for split in ["train", "val"]:
        loss_list = []
        for _ in range(20):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            loss_list.append(loss.item())
        losses[split] = sum(loss_list) / len(loss_list)
    model.train()
    return losses


model = GPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

best_val_loss = float("inf")
patience = 5
counter = 0

for step in range(max_iters):

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0:

        losses = estimate_loss()
        print(f"step {step}, train {losses['train']:.4f}, val {losses['val']:.4f}")

        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            counter = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            counter += 1

        if counter >= patience:
            print("🚀 Early stopping triggered!")
            break


# ================================
# GENERATION
# ================================

start_token = tokenizer.bos_token_id or tokenizer.eos_token_id
context = torch.tensor([[tokenizer.eos_token_id]], device=device)

generated = model.generate(
    context,
    max_new_tokens=150,
    temperature=0.8,
    top_k=50
)
print("\n=== GENERATED TEXT ===\n")
print(decode(generated[0].tolist()))




[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!



[INFO] Loaded dataset
Total characters: 538360726
Sample raw text:
  
 = Valkyria Chronicles III = 
 
 Senjō no Valkyria 3 : <unk> Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStatio

[INFO] After fixing special tokens
 
 = Valkyria Chronicles III = 
 
 Senjō no Valkyria 3 : UNK Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role - playing video game developed by Sega and Media.Vision for the PlayStation Po

[INFO] After removing section titles
 
   
 
 Senjō no Valkyria 3 : UNK Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role - playing video game developed by Sega and Media.Vision for

Token indices sequence length is longer than the specified maximum sequence length for this model (21503 > 1024). Running this sequence through the model will result in indexing errors


Encoding text in chunks...

===== DATASET ANALYSIS =====
Total Words           : 854,466
Total Tokens (BPE)    : 1,069,286
Unique Words          : 92,632
Unique Tokens         : 35,425
Avg Tokens per Word   : 1.25
Avg Word Length       : 4.85
step 0, train 10.2280, val 10.2331
step 500, train 6.0016, val 6.3493
step 1000, train 5.1662, val 6.0209
step 1500, train 4.4067, val 5.9377
step 2000, train 3.7639, val 6.0443
step 2500, train 3.0490, val 6.2611
step 3000, train 2.3807, val 6.5874
step 3500, train 1.8686, val 6.9938
step 4000, train 1.4193, val 7.3179
🚀 Early stopping triggered!

=== GENERATED TEXT ===

<|endoftext|>. This was the first ever to create military units: the French people, with respect to distort the varietal of the spirits of Palestine. In 1970, the regiment's army was transferred to the Middle Preoccupied Belester in the region, with a squadron of French explorer Aurel Phille and French law, and to raise money for the attack on 21 December. Both ships were rated 1